# Resident Reintegration Readiness Scorer

## 1. Problem framing
Case managers need **prioritization** for residents approaching reintegration—who may need more visits, planning, or resources in the next window. **Stakeholders:** social workers, clinical supervisor, program director. **Predictive vs explanatory:** **Primary predictive**—out-of-sample discrimination for a binary near-term outcome; **secondary explanatory**—logistic-style interpretation where exported for transparency. **Justification:** Operations require **ranking** under capacity constraints (prediction); governance requires **interpretable** drivers (explanation)—textbook dual use.

## 2. Data acquisition, preparation & exploration
Residents joined to **process_recordings**, **home_visitations**, **education_records**, **health_wellbeing_records**, etc., at observation dates with **no future leakage**. Modules: `data_prep.py`, `feature_engineering.py`. Notebook steps profile counts/missingness. **Reproducible pipeline:** `run_all.py`, `export_artifacts.py`, `score_all_current_residents.py`.

## 3. Modeling & feature selection
`train_predictive` / `train_explanatory` and final calibration in `model_finalize.py` as applicable. Features encode engagement intensity, risk flags, and trajectory summaries. Hyperparameters and CV in training scripts.

## 4. Evaluation & interpretation
ROC/PR, calibration, Brier where applicable; stratified splits. **False negative** (miss a struggling resident) risks **poor transition**; **false positive** (over-alert) consumes **scarce staff time**—asymmetric costs favor recall for high-risk bands with human triage.

## 5. Causal and relationship analysis
Importances show **which recorded factors co-occur** with modeled readiness—not proof that visits *cause* success (reverse causality, selection into services). Discuss limitations honestly; use scores as **decision support** only.

## 6. Deployment notes
**Artifacts:** `serialized_models/` + `outputs/*.json`. **Backend:** `refresh_ml_artifacts.py --reintegration-only` → `App_Data/ml/reintegration/`. **.NET:** `MlResidentsController` and related services read JSON. **UI:** `ResidentsListPage.tsx` ML columns; `AdminHomePage` widget; `frontend/src/lib/mlApi.ts` (`getResidentPriority`, `getResidentCurrentScores`).

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

def _find_repo_root(start: Path) -> Path:
    p = start.resolve()
    for d in [p, *p.parents]:
        if (d / "ml-pipelines").is_dir() and (d / "data" / "lighthouse_csv_v7").is_dir():
            return d
    raise FileNotFoundError(
        "Could not find repo root (need ml-pipelines/ and data/lighthouse_csv_v7/). "
        f"cwd={p}"
    )

REPO_ROOT = _find_repo_root(Path.cwd())
ML_PIPELINES_DIR = REPO_ROOT / "ml-pipelines"
ML_PIPELINE = ML_PIPELINES_DIR
DATA_DIR = REPO_ROOT / "data" / "lighthouse_csv_v7"
if str(ML_PIPELINES_DIR) not in sys.path:
    sys.path.insert(0, str(ML_PIPELINES_DIR))

print("ML_PIPELINE:", ML_PIPELINE)
print("DATA_DIR:", DATA_DIR)

## Step 1 — Load & profile tables

In [ ]:
from ml_pipeline_reintegration_readiness_scorer.data_prep import load_tables, parse_datetime_columns, profile_tables

tables = parse_datetime_columns(load_tables(DATA_DIR))
prof = profile_tables(tables)
for name, p in prof.items():
    print(f"{name}: rows={p['rows']}")
    print("  missing (top):", list(p["missing_top"].items())[:5])

## Step 2–3 — Snapshot dataset & label

In [ ]:
from ml_pipeline_reintegration_readiness_scorer.feature_engineering import build_snapshot_frame, global_data_cutoff

print("Data cutoff:", global_data_cutoff(tables))
df, meta = build_snapshot_frame(tables)
print(json.dumps(meta, indent=2))
display(df[df.columns[:8]].head())
print(df["reintegration_success_next_60_days"].value_counts())

## Step 6–8 — Train, evaluate, coefficients

In [ ]:
from ml_pipeline_reintegration_readiness_scorer.export_artifacts import export_all

info = export_all(data_dir=DATA_DIR)
print(json.dumps(info, indent=2, default=str))

In [ ]:
from ml_pipeline_reintegration_readiness_scorer import config

with open(config.SERIALIZED_DIR / "reintegration_readiness_metadata.json", encoding="utf-8") as f:
    m = json.load(f)
print("Best model:", m["best_model"])
print("Selected threshold (OOF-tuned):", m.get("selected_threshold"))
# Metadata now reports holdout metrics at default 0.5 and at the tuned threshold (see README)
print("Holdout @ threshold 0.5:\n", json.dumps(m.get("holdout_metrics_threshold_0.5"), indent=2, default=str))
print("Holdout @ tuned threshold:\n", json.dumps(m.get("holdout_metrics_tuned_threshold"), indent=2, default=str))
coef = pd.read_csv(config.OUTPUTS_DIR / "explanatory_logistic_coefficients.csv")
display(coef.head(15))